# Clasificación binaria de TipoEscena

Este notebook replica el flujo de `neuroflex_3classes_model` con una variable objetivo distinta: **`TipoEscena`**, que contiene dos categorías (`1` y `2`); en el archivo aumentado, los valores con ruido se asignan a la clase más cercana usando el punto medio `1.5`. Se comparan varios clasificadores y se elige el mejor usando F1 ponderado en validación.

Se excluyen variables de identidad y la variable objetivo de las entradas para reducir fuga de información.


In [ ]:
# Importaciones y configuración
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
DATA_PATH = Path('neuroflex_dataset_augmented.csv')
print('Configuración lista. Objetivo: TipoEscena')


In [ ]:
# Cargar y explorar los datos
df = pd.read_csv(DATA_PATH)
print(f'Dimensiones: {df.shape}')
print('Columnas:', df.columns.tolist())
print('Nulos por columna (solo columnas con nulos):')
print(df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False))

if 'age' in df.columns:
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace(r'^\s*$', np.nan, regex=True)


In [ ]:
# Definir variable objetivo y revisar sus clases
TARGET = 'TipoEscena'
if TARGET not in df.columns:
    raise KeyError(f'No se encontró la columna objetivo {TARGET!r}')

df[TARGET] = pd.to_numeric(df[TARGET], errors='coerce')
# El dataset aumentado agregó ruido a esta columna; asignar cada valor a la clase original más cercana.
df[TARGET] = pd.to_numeric(df[TARGET], errors='coerce')
df[TARGET] = (df[TARGET] >= 1.5).astype(int)
df = df.dropna(subset=[TARGET]).copy()
class_counts = df[TARGET].value_counts().sort_index()
print('Distribución de TipoEscena:')
print(class_counts)
print('Proporciones:')
print((class_counts / class_counts.sum()).round(4))

ax = class_counts.plot(kind='bar', color=['steelblue', 'coral'], edgecolor='black', figsize=(7, 4))
ax.set_title('Distribución de la variable objetivo: TipoEscena')
ax.set_xlabel('TipoEscena')
ax.set_ylabel('Frecuencia')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Preparar predictores: descartar objetivo e identificadores
# Precision y métricas de desempeño se excluyen para evitar que el modelo use
# resultados de la sesión como atajo para inferir TipoEscena.
exclude_cols = [
    TARGET, 'timestamp', 'session_id', 'date', 'name', 'rut', 'patient_updated_at',
    'Precision', 'CantAciertasTotales', 'ObjetosInteractuadosCorrectamente'
]
feature_cols = [c for c in df.columns if c not in exclude_cols]
X = df[feature_cols].copy()
y = df[TARGET].copy()

# Convertir columnas numéricas que hayan quedado como texto.
for col in X.columns:
    if X[col].dtype == 'object' or str(X[col].dtype) == 'string':
        converted = pd.to_numeric(X[col], errors='coerce')
        if converted.notna().mean() >= 0.8:
            X[col] = converted

print('Número de predictores:', len(feature_cols))
print('Predictores:', feature_cols)


In [ ]:
# Separar train, validación y test manteniendo proporción de clases
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)
print('Train:', X_train.shape, ' Validación:', X_val.shape, ' Test:', X_test.shape)
for nombre, serie in [('Train', y_train), ('Validación', y_val), ('Test', y_test)]:
    print(f'\n{nombre}:')
    print(serie.value_counts().sort_index())


In [ ]:
# Preprocesamiento numérico y categórico
categorical_cols = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                          ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols),
    ],
    remainder='drop'
)
print(f'Variables numéricas: {len(numeric_cols)} | categóricas: {len(categorical_cols)}')


In [ ]:
# Entrenar y comparar modelos en validación
models = {
    'LogisticRegression': LogisticRegression(max_iter=5000, class_weight='balanced', random_state=SEED),
    'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=SEED),
    'ExtraTrees': ExtraTreesClassifier(n_estimators=300, class_weight='balanced', random_state=SEED),
    'GradientBoosting': GradientBoostingClassifier(random_state=SEED),
    'SVC': SVC(class_weight='balanced', random_state=SEED),
}
results = []
pipelines = {}
for name, estimator in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)
    row = {
        'model': name,
        'accuracy': accuracy_score(y_val, pred),
        'precision_weighted': precision_score(y_val, pred, average='weighted', zero_division=0),
        'recall_weighted': recall_score(y_val, pred, average='weighted', zero_division=0),
        'f1_weighted': f1_score(y_val, pred, average='weighted', zero_division=0),
    }
    results.append(row)
    pipelines[name] = pipe
    print(f"{name}: accuracy={row['accuracy']:.4f}, F1 ponderado={row['f1_weighted']:.4f}")
results_df = pd.DataFrame(results).sort_values('f1_weighted', ascending=False).reset_index(drop=True)
display(results_df)


In [ ]:
# Comparar métricas de validación
plot_df = results_df.set_index('model')[['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']]
ax = plot_df.plot(kind='bar', figsize=(12, 5), ylim=(0, 1), rot=25)
ax.set_title('Comparación de clasificadores en validación')
ax.set_ylabel('Puntaje')
ax.set_xlabel('Modelo')
plt.legend(['Accuracy', 'Precision ponderado', 'Recall ponderado', 'F1 ponderado'])
plt.tight_layout()
plt.show()


In [ ]:
# Evaluación final del modelo seleccionado en el conjunto de test
best_model_name = results_df.loc[0, 'model']
best_pipe = pipelines[best_model_name]
y_test_pred = best_pipe.predict(X_test)
print('Modelo seleccionado:', best_model_name)
print(f'Accuracy: {accuracy_score(y_test, y_test_pred):.4f}')
print(f'Precision ponderado: {precision_score(y_test, y_test_pred, average="weighted", zero_division=0):.4f}')
print(f'Recall ponderado: {recall_score(y_test, y_test_pred, average="weighted", zero_division=0):.4f}')
print(f'F1 ponderado: {f1_score(y_test, y_test_pred, average="weighted", zero_division=0):.4f}')
print('\nReporte por clase:')
print(classification_report(y_test, y_test_pred, labels=sorted(y.unique()), zero_division=0, digits=4))


In [ ]:
# Matriz de confusión
labels = [0, 1]
class_names = ['TipoEscena 1', 'TipoEscena 2']
cm = confusion_matrix(y_test, y_test_pred, labels=labels)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f'Matriz de confusión — {best_model_name} (test)')
plt.xlabel('TipoEscena predicho')
plt.ylabel('TipoEscena real')
plt.tight_layout()
plt.show()


## Conclusiones

El objetivo de este notebook es predecir `TipoEscena` (`1` o `2`) a partir de características demográficas y tiempos de interacción. La tabla de validación permite elegir un algoritmo y la evaluación final informa su desempeño en test.

Interpreta el resultado considerando el tamaño y el posible desbalance del conjunto de datos. Las variables de desempeño de la sesión se excluyeron de los predictores para reducir fuga de información.
